# 🎯 Phone Detection Training - YOLOv8 on GPU

Train `phone_v3` model trên **GPU T4 miễn phí** của Google Colab.

## Hướng dẫn:
1. Ở máy local, chạy lệnh PowerShell ở **Bước 0** để tạo file zip dataset
2. Mở notebook này trên Google Colab
3. Vào **Runtime → Change runtime type → GPU (T4)**
4. Chạy từng cell theo thứ tự
5. Upload file `phone_detection_v3.zip` khi được yêu cầu
6. Đợi training hoàn thành (~5-10 phút cho 40 epochs trên T4)
7. Download `best.pt` về máy

## Bước 0: Tạo file ZIP dataset (chạy ở MÁY LOCAL)

**Mở PowerShell** trên máy local và chạy lệnh sau:

```powershell
cd D:\HUNRE\NCKH2025\cheating-detection\cheating-detection\dataset
Compress-Archive -Path phone_detection_v3 -DestinationPath phone_detection_v3.zip -Force
```

File zip sẽ nằm tại `D:\HUNRE\NCKH2025\cheating-detection\cheating-detection\dataset\phone_detection_v3.zip` (~250MB)

---
## ⬇️ Các cell dưới đây chạy trên GOOGLE COLAB

In [ ]:
# ============================================================
# Bước 1: Kiểm tra GPU
# ============================================================
!nvidia-smi
import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

if not torch.cuda.is_available():
    print("\n⚠️ GPU chưa được bật! Vào Runtime → Change runtime type → GPU (T4)")
    raise RuntimeError("No GPU detected!")

In [ ]:
# ============================================================
# Bước 2: Cài đặt ultralytics
# ============================================================
!pip install -q ultralytics
from ultralytics import YOLO
print("✅ Ultralytics installed successfully!")

In [ ]:
# ============================================================
# Bước 3: Upload dataset zip
# ============================================================
from google.colab import files
import os

print("📂 Hãy upload file phone_detection_v3.zip (~250MB)...")
print("   (File này được tạo ở Bước 0 trên máy local)\n")

uploaded = files.upload()

# Giải nén
!unzip -q -o phone_detection_v3.zip -d /content/dataset/

# Kiểm tra
train_count = len(os.listdir('/content/dataset/phone_detection_v3/images/train'))
val_count = len(os.listdir('/content/dataset/phone_detection_v3/images/val'))
print(f"\n✅ Dataset uploaded: {train_count} train images, {val_count} val images")

In [ ]:
# ============================================================
# Bước 3b (Tùy chọn): Upload từ Google Drive thay vì upload trực tiếp
# ============================================================
# Nếu bạn đã upload zip lên Google Drive, uncomment và chạy cell này
# thay cho Bước 3 ở trên.

# from google.colab import drive
# drive.mount('/content/drive')
#
# # Thay đổi đường dẫn cho đúng vị trí file trên Drive của bạn
# DRIVE_ZIP = '/content/drive/MyDrive/phone_detection_v3.zip'
#
# !unzip -q -o "$DRIVE_ZIP" -d /content/dataset/
# import os
# train_count = len(os.listdir('/content/dataset/phone_detection_v3/images/train'))
# val_count = len(os.listdir('/content/dataset/phone_detection_v3/images/val'))
# print(f"✅ Dataset from Drive: {train_count} train, {val_count} val images")

In [ ]:
# ============================================================
# Bước 4: Cập nhật data.yaml cho Colab paths
# ============================================================
data_yaml = """
names:
  0: phone
nc: 1
train: /content/dataset/phone_detection_v3/images/train
val: /content/dataset/phone_detection_v3/images/val
""".strip()

with open('/content/dataset/phone_detection_v3/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml updated for Colab paths")
print(data_yaml)

In [ ]:
# ============================================================
# Bước 5: 🚀 TRAIN MODEL
# ============================================================
# Trên GPU T4: ~10-20 giây/epoch → ~10 phút cho 40 epochs
# (So với ~7 phút/epoch trên CPU = nhanh hơn ~20-40x)
# ============================================================

from ultralytics import YOLO
import time

model = YOLO('yolov8n.pt')  # Download pretrained model

start = time.time()

results = model.train(
    data='/content/dataset/phone_detection_v3/data.yaml',
    epochs=40,
    patience=10,
    imgsz=640,
    batch=16,          # Tăng batch size vì GPU có đủ VRAM
    workers=2,
    device=0,          # GPU 0
    seed=42,
    project='/content/runs',
    name='phone_v3',
    exist_ok=True,
    cache=True,        # Cache ảnh vào RAM để tăng tốc
    amp=True,          # Mixed precision training
    plots=True,
    verbose=True,
)

elapsed = time.time() - start
print(f"\n🎉 Training hoàn thành trong {elapsed/60:.1f} phút!")

In [ ]:
# ============================================================
# Bước 6: Xem kết quả training
# ============================================================
from IPython.display import Image, display
import os

results_dir = '/content/runs/phone_v3'

# Hiển thị các biểu đồ
for img_name in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = os.path.join(results_dir, img_name)
    if os.path.exists(img_path):
        print(f"\n📊 {img_name}:")
        display(Image(filename=img_path, width=800))

In [ ]:
# ============================================================
# Bước 7: Validate model
# ============================================================
best_model = YOLO('/content/runs/phone_v3/weights/best.pt')

metrics = best_model.val(
    data='/content/dataset/phone_detection_v3/data.yaml',
    imgsz=640,
    batch=16,
    device=0,
)

print("\n" + "="*50)
print("📊 KẾT QUẢ VALIDATION")
print("="*50)
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")
print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print("="*50)

In [ ]:
# ============================================================
# Bước 8: Download model weights về máy
# ============================================================
from google.colab import files
import shutil

# Tạo zip chứa cả best.pt, last.pt và results
shutil.make_archive('/content/phone_v3_results', 'zip', '/content/runs/phone_v3')

print("📦 Đang download phone_v3_results.zip...")
print("   Bao gồm: best.pt, last.pt, results.csv, biểu đồ training")
print("\n   Sau khi download, copy best.pt vào:")
print("   D:\\HUNRE\\NCKH2025\\cheating-detection\\cheating-detection\\models\\phone_v3\\weights\\")

files.download('/content/phone_v3_results.zip')

In [ ]:
# ============================================================
# Bước 8b (Tùy chọn): Lưu vào Google Drive thay vì download
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = '/content/drive/MyDrive/phone_v3_results'
# shutil.copytree('/content/runs/phone_v3', dest, dirs_exist_ok=True)
# print(f"✅ Saved to Google Drive: {dest}")